# Sequence-to-Sequence (Summarization & Translation)

> Encoder-decoder generation workflows with `Seq2SeqLearner`.

Use this notebook for summarization, translation, and other text-to-text tasks.

## Notebook Metadata

`nbdev` export target for this notebook.

In [ ]:
#| default_exp text.seq2seq


## Setup

Import dependencies.

In [ ]:
import warnings

warnings.filterwarnings('ignore')

import pandas as pd

from blurr.text.seq2seq import Seq2SeqLearner


## Quick Start

Train a tiny summarizer from a DataFrame and generate one summary.

In [ ]:
df = pd.DataFrame({
    'article': [
        'The city council approved a new affordable housing plan after months of debate.',
        'Scientists reported encouraging results from an early-stage malaria vaccine trial.',
        'A major storm is expected to make landfall late Friday with heavy rainfall.',
        'The startup raised funding to expand its renewable energy storage platform.',
    ],
    'summary': [
        'City council approves affordable housing plan.',
        'Early malaria vaccine trial shows promising results.',
        'Major storm expected Friday with heavy rain.',
        'Startup raises funding for energy storage expansion.',
    ],
})

learn = Seq2SeqLearner.from_df(
    df,
    model_name='t5-small',
    input_col='article',
    target_col='summary',
    prefix='summarize: ',
    max_source_length=128,
    max_target_length=48,
    bs=2,
)

learn.fit_one_cycle(1, 2e-4)
quick_gen = learn.blurr_generate(
    'Local officials announced expanded public transit service beginning next month.',
    prefix='summarize: ',
    max_new_tokens=40,
)[0]
quick_gen


**Expected output**

A dictionary with `generated_text` containing the model-generated summary.

## Generation Controls

Tune decoding behavior using Hugging Face generation kwargs.

In [ ]:
gen_examples = learn.blurr_generate(
    [
        'Researchers released a benchmark comparing open-source language models.',
        'The team unveiled a redesigned mobile app with offline support.',
    ],
    prefix='summarize: ',
    max_new_tokens=48,
    num_beams=4,
    no_repeat_ngram_size=3,
)

for ex in gen_examples:
    print(ex['generated_text'])


## Advanced: Translation-style Inputs

For translation datasets with nested columns, pass explicit `input_col` and `target_col`; `from_hf` handles common nested structures (e.g. `translation.de`, `translation.en`).

In [ ]:
# Example shape (not executed here):
# learn_mt = Seq2SeqLearner.from_hf(
#     'wmt16', 'de-en',
#     model_name='Helsinki-NLP/opus-mt-de-en',
#     input_col='de',
#     target_col='en',
#     bs=8,
# )
# learn_mt.fit_one_cycle(1, 2e-5)


## API Reference

### Main class
- `Seq2SeqLearner`

### Construction methods
- `from_df(df, model_name, ...)`
- `from_hf(dataset, subset=None, model_name=..., ...)`

### Inference
- `blurr_generate(texts, prefix='', max_new_tokens=..., num_beams=..., **gen_kwargs)`

### Metrics
- ROUGE metrics are added via `Seq2SeqMetricsCallback` when `rouge-score` is installed.
